# E2 · charm etiketleri / nnU-Net — Colab eğitimi

**Veri:** K1'in `handoff_k2_e2.zip` paketi, `Dataset002_E2Charm`. Sabit **67 eğitim + 16 doğrulama**
(IXI 23/6,
MultiAxial 44/10), tek T1 kanalı,
0–5 etiketleri, `3d_fullres`, yalnız **fold 0**. Test ve ayrılmış vakalar pakette yok; defter bunu da denetler.

Ayarlar E1_L4_001 ile aynıdır: aynı sürümler, seed 42, her epoch checkpoint, 5 epoch kontrol ve 100 epoch ön model.
Böylece E1 ile E2 arasındaki fark veriden gelir.

1. Drive'da Berke'nin `TMS` klasöründeki `handoff_k2_e2.zip` dosyasına sağ tıkla → **Düzenle → Kısayol ekle** →
   **Drive'ım** altında `TMS_E2` adlı bir klasör oluşturup onu seç. (Kopyalamak da olur; kısayol kotanı harcamaz.)
2. Colab'da bu defteri aç; çalışma zamanı olarak GPU seç (L4 önerilir; en az 12 GB VRAM).
3. Hücreleri sırayla çalıştır. Önizlemeyi ve veri notlarını incelemeden eğitim açılmaz.

**Varsayılan ayarlar eğitim başlatmaz.** GPU oturumu açık kaldığı sürece birim harcar.
Sonuçlar/checkpoint'ler Drive'da, veri ve ön işleme Colab'ın yerel diskinde tutulur.


In [ ]:
# Ayarlar. Devam ederken aynı deney kimliğini koru.
from pathlib import Path
import json, os, re, shutil, subprocess, sys, time

ARCHIVE = Path("/content/drive/MyDrive/TMS_E2/handoff_k2_e2.zip")
RESULTS_BASE = Path("/content/drive/MyDrive/TMS_E2/runs")
EXPERIMENT_ID = "E2_L4_001"
CANONICALIZE_TO_RAS = True  # IXI PSR saklanıyor, MultiAxial RAS; nnU-Net döndürmez. Açıklama 3. bölümde.
ACKNOWLEDGE_E2_DATA = False  # Önizlemeyi ve veri notlarını inceledikten sonra True yap.
RUN_5_EPOCH_PILOT = False
RUN_100_EPOCH_PRELIMINARY = False
RESUME_PILOT = False
RESUME_PRELIMINARY = False
RELEASE_RUNTIME_WHEN_DONE = False  # True: 100 epoch ve tam doğrulama bitince GPU oturumunu bırak.
CU_PER_HOUR = 0.0  # Açık oturumun gösterilen birim/saat değeri; 0 = bilinmiyor.
AVAILABLE_CU = 0.0
AUGMENTATION_WORKERS = 2
assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", EXPERIMENT_ID)
assert 1 <= AUGMENTATION_WORKERS <= 4 and CU_PER_HOUR >= 0 and AVAILABLE_CU >= 0
WORK = Path("/content/e2_work")
TOOLS = Path("/content/e2_tools")
VENV = Path("/content/e2-venv")
PYTHON = VENV / "bin/python"
DATA = WORK / "handoff"
RAS_RAW = WORK / "raw_ras"
PREPROCESSED = WORK / "nnUNet_preprocessed"
RESULTS = RESULTS_BASE / EXPERIMENT_ID


## 1 · Drive bağlantısı
Paylaşılan klasörler Colab'da `MyDrive` altında görünmez; bu yüzden 1. adımdaki kısayol gerekir.


In [ ]:
assert sys.platform == "linux", "Bu defteri Google Colab'ın Linux oturumunda aç."
from google.colab import drive
drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Kalıcı Drive bağlantısı kurulamadı."
assert ARCHIVE.is_file(), (f"Paket bulunamadı: {ARCHIVE}\n"
    "Berke'nin TMS klasöründeki handoff_k2_e2.zip için Drive'ım/TMS_E2 altına kısayol ekle ya da yolu düzelt.")
WORK.mkdir(parents=True, exist_ok=True)
TOOLS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
assert shutil.disk_usage("/content").free > 40 * 1024**3, "Kurulum, veri ve ön işleme için en az 40 GiB boş yer gerekli."


## 2 · Ayrı, sürümü sabitlenmiş eğitim ortamı
E1 ile aynı kilitli bağımlılıklar (Python 3.11, nnU-Net 2.8.1, PyTorch 2.6.0 + CUDA 12.4).
Gömülü yardımcılar proje deposundan üretilmiştir; ZIP içindeki hiçbir dosya kod olarak çalıştırılmaz.


In [ ]:
# Gömülü, sürümlü yardımcılar.
EMBEDDED = {'e2_helpers.py': '"""E2 Colab helpers: check K1\'s handoff, put every case in one axis order, build training commands.\n\nImporting this module never starts training. Functions that read NIfTI import nibabel lazily, so the\narchive checks also run in Colab\'s own kernel.\n"""\n\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\n\nDATASET = "Dataset002_E2Charm"\nDATASET_ID = "2"\nLABELS = {"background": 0, "skin": 1, "skull": 2, "csf": 3, "gray_matter": 4, "white_matter": 5}\nTRAINERS = ("nnUNetTrainer_E2Smoke", "nnUNetTrainer_E2Preliminary")\nMAX_ARCHIVE_BYTES = 20_000_000_000\nGRID_ATOL_MM = 1e-3\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _layout(names):\n    """Where dataset.json and splits_final.json sit inside the archive (a top folder is allowed)."""\n    marker = f"nnUNet_raw/{DATASET}/dataset.json"\n    found = [name for name in names if name == marker or name.endswith("/" + marker)]\n    require(len(found) == 1, f"Arşivde tam bir tane {marker} olmalı; bulunan: {len(found)}.")\n    prefix = found[0][: -len(marker)]\n    preferred = f"{prefix}nnUNet_preprocessed/{DATASET}/splits_final.json"\n    splits = [preferred] if preferred in names else [n for n in names if PurePosixPath(n).name == "splits_final.json"]\n    require(len(splits) == 1, f"Arşivde tek bir splits_final.json bulunmalı; bulunan: {len(splits)}.")\n    return {"dataset": f"{prefix}nnUNet_raw/{DATASET}", "raw": f"{prefix}nnUNet_raw", "splits": splits[0]}\n\n\ndef extract_handoff(archive_path, destination, expected_sha256):\n    """Extract K1\'s archive once, after its SHA-256 matches the published value; return its layout."""\n    archive_path, destination = Path(archive_path), Path(destination).resolve()\n    digest = sha256(archive_path)\n    require(digest == expected_sha256, f"Arşivin SHA-256 değeri yayımlanan pakete uymuyor: {digest}")\n    marker, layout_file = destination / ".archive_sha256", destination / ".layout.json"\n    if destination.exists():\n        require(marker.is_file() and marker.read_text().strip() == digest,\n                "Bu klasör başka bir arşivden açılmış. Yeni bir hedef klasör seç.")\n        layout = json.loads(layout_file.read_text())\n        return {key: destination / value for key, value in layout.items()}\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive_path) as archive:\n        infos = archive.infolist()\n        # Archives made by Windows PowerShell 5.1 store "\\" separators; read them as "/".\n        entries = [(info, info.filename.replace("\\\\", "/")) for info in infos]\n        files = [(info, name) for info, name in entries if not name.endswith("/")]\n        names = [name for _, name in files]\n        require(len(names) == len(set(names)), "Arşivde aynı yol iki kez geçiyor.")\n        require(sum(info.file_size for info in infos) <= MAX_ARCHIVE_BYTES, "Arşiv beklenenden çok büyük.")\n        for info, name in entries:\n            path = PurePosixPath(name)\n            require(not path.is_absolute() and ".." not in path.parts and ":" not in name,\n                    f"Güvensiz arşiv yolu: {info.filename}")\n            require(not stat.S_ISLNK(info.external_attr >> 16), f"Arşivde bağlantı (symlink) var: {info.filename}")\n        layout = _layout(names)\n        with tempfile.TemporaryDirectory(prefix="e2-extract-", dir=destination.parent) as folder:\n            temporary = Path(folder) / "content"\n            for info, name in files:\n                output = temporary / name\n                output.parent.mkdir(parents=True, exist_ok=True)\n                with archive.open(info) as source, output.open("xb") as target:\n                    shutil.copyfileobj(source, target, length=1024 * 1024)\n            (temporary / ".archive_sha256").write_text(digest + "\\n")\n            (temporary / ".layout.json").write_text(json.dumps(layout, indent=2) + "\\n")\n            temporary.rename(destination)\n    return {key: destination / value for key, value in layout.items()}\n\n\ndef _cases(dataset):\n    return sorted(p.name[: -len("_0000.nii.gz")] for p in (Path(dataset) / "imagesTr").glob("*_0000.nii.gz"))\n\n\ndef dataset_fingerprint(dataset, splits):\n    """One hash over every file name and content in the dataset folder plus the split."""\n    dataset = Path(dataset)\n    lines = [f"{sha256(path)}  {path.relative_to(dataset).as_posix()}"\n             for path in sorted(p for p in dataset.rglob("*") if p.is_file())]\n    lines.append(f"{sha256(splits)}  splits_final.json")\n    return hashlib.sha256("\\n".join(lines).encode()).hexdigest()\n\n\ndef audit(dataset, splits, contract):\n    """Technical checks against the contract; returns a report. Not an anatomical review."""\n    import nibabel as nib\n    import numpy as np\n\n    dataset, splits = Path(dataset), Path(splits)\n    for name in ("imagesTs", "labelsTs"):\n        require(not (dataset / name).exists(), f"Test klasörü eğitim paketine girmemeli: {name}")\n    metadata = json.loads((dataset / "dataset.json").read_text())\n    require(metadata.get("labels") == LABELS, "Etiket sözleşmesi 0–5 (deri, kafatası, BOS, GM, WM) değil.")\n    require(metadata.get("channel_names") == {"0": "T1"}, "Tek T1 kanalı bekleniyordu.")\n    require(metadata.get("file_ending") == ".nii.gz", ".nii.gz dosyaları bekleniyordu.")\n    folds = json.loads(splits.read_text())\n    require(len(folds) == 1 and set(folds[0]) == {"train", "val"}, "Tek bir sabit train/val bölmesi bekleniyordu.")\n    train, val = folds[0]["train"], folds[0]["val"]\n    require(sorted(train) == sorted(contract["train"]) and sorted(val) == sorted(contract["val"]),\n            "Bölme, K1\'in atamasından hesaplanan listeyle aynı değil.")\n    require(not set(train) & set(val), "Bir vaka hem eğitimde hem doğrulamada.")\n    cases = set(train) | set(val)\n    require(metadata.get("numTraining") == len(cases), "dataset.json numTraining bölmeyle uyuşmuyor.")\n    images = {p.name for p in (dataset / "imagesTr").iterdir()}\n    labels = {p.name for p in (dataset / "labelsTr").iterdir()}\n    require(images == {f"{case}_0000.nii.gz" for case in cases}, "imagesTr\'de eksik, fazla ya da yanlış adlı dosya var.")\n    require(labels == {f"{case}.nii.gz" for case in cases}, "labelsTr\'de eksik, fazla ya da yanlış adlı dosya var.")\n    report = {"dataset": DATASET, "train_count": len(train), "validation_count": len(val), "cases": [],\n              "axis_codes": {}, "warnings": []}\n    for case in sorted(cases):\n        image = nib.load(dataset / "imagesTr" / f"{case}_0000.nii.gz")\n        label = nib.load(dataset / "labelsTr" / f"{case}.nii.gz")\n        require(image.ndim == 3 and label.shape == image.shape, f"{case}: 3B görüntü ve aynı boyutta etiket bekleniyordu.")\n        require(np.allclose(image.affine, label.affine, rtol=0, atol=GRID_ATOL_MM), f"{case}: görüntü ve etiket aynı ızgarada değil.")\n        values = np.asanyarray(label.dataobj)\n        require(np.all(values == np.round(values)), f"{case}: etiketler tam sayı değil.")\n        present = sorted(int(v) for v in np.unique(values))\n        require(set(present) <= set(LABELS.values()), f"{case}: 0–5 dışında etiket var: {present}")\n        require(np.isfinite(np.asanyarray(image.dataobj)).all(), f"{case}: görüntüde sonlu olmayan değer var.")\n        codes = "".join(nib.aff2axcodes(image.affine))\n        report["axis_codes"][codes] = report["axis_codes"].get(codes, 0) + 1\n        report["cases"].append({"case": case, "split": "train" if case in train else "val", "shape": list(image.shape),\n                                "axis_codes": codes, "units": image.header.get_xyzt_units()[0],\n                                "zooms": [round(float(z), 4) for z in image.header.get_zooms()[:3]],\n                                "labels_present": present})\n    if len(report["axis_codes"]) > 1:\n        report["warnings"].append(f"Vakalar farklı eksen sıralarında saklanmış: {report[\'axis_codes\']}. "\n                                  "nnU-Net görüntüyü döndürmez.")\n    report["dataset_sha256"] = dataset_fingerprint(dataset, splits)\n    report["split_sha256"] = sha256(splits)\n    return report\n\n\ndef _to_ras(path, out_path):\n    """Permute and flip axes toward RAS without interpolation; returns the source axis codes."""\n    import nibabel as nib\n    from nibabel import orientations\n    import numpy as np\n\n    image = nib.load(path)\n    data = np.asanyarray(image.dataobj)\n    transform = orientations.io_orientation(image.affine)\n    moved = orientations.apply_orientation(data, transform).copy()\n    affine = image.affine @ orientations.inv_ornt_aff(transform, data.shape)\n    result = nib.Nifti1Image(moved, affine)\n    result.set_data_dtype(data.dtype)\n    result.set_sform(affine, code=1)\n    result.set_qform(affine, code=1)\n    result.header.set_xyzt_units(image.header.get_xyzt_units()[0])\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    nib.save(result, str(out_path))\n    return "".join(nib.aff2axcodes(image.affine)), transform.tolist()\n\n\ndef write_ras_dataset(dataset, raw_out):\n    """Copy the dataset into raw_out with every image and label in RAS order; returns the change log.\n\n    Voxel values are never interpolated: each case is an exact axis permutation and flip of the original,\n    and the world position of every voxel is unchanged."""\n    dataset, target = Path(dataset), Path(raw_out) / DATASET\n    require(not target.exists(), f"{target} zaten var; yeni bir klasör seç.")\n    partial = Path(raw_out) / f".partial_{DATASET}"\n    if partial.exists():  # left by an interrupted run; never used for training\n        shutil.rmtree(partial)\n    changes = {}\n    for case in _cases(dataset):\n        image_codes, transform = _to_ras(dataset / "imagesTr" / f"{case}_0000.nii.gz",\n                                         partial / "imagesTr" / f"{case}_0000.nii.gz")\n        label_codes, label_transform = _to_ras(dataset / "labelsTr" / f"{case}.nii.gz", partial / "labelsTr" / f"{case}.nii.gz")\n        require(image_codes == label_codes and transform == label_transform, f"{case}: görüntü ve etiket aynı dönüşümü almadı.")\n        changes[case] = {"source_axis_codes": image_codes, "to_ras": transform}\n    shutil.copyfile(dataset / "dataset.json", partial / "dataset.json")\n    partial.rename(target)\n    return changes\n\n\ndef install_split(splits, preprocessed):\n    target = Path(preprocessed) / DATASET / "splits_final.json"\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if target.exists():\n        require(json.loads(target.read_text()) == json.loads(Path(splits).read_text()),\n                "Kayıtlı bölme farklı; üzerine yazılmıyor.")\n    else:\n        shutil.copyfile(splits, target)\n    return sha256(target)\n\n\ndef training_command(executable, trainer, fold_directory, resume=False):\n    require(trainer in TRAINERS, "Beklenmeyen E2 eğiticisi.")\n    folder = Path(fold_directory)\n    if resume:\n        checkpoints = [folder / f"checkpoint_{name}.pth" for name in ("final", "latest", "best")]\n        checkpoint = next((path for path in checkpoints if path.is_file()), None)\n        require(checkpoint is not None, "Devam istendi ama checkpoint yok; sessizce baştan başlatılmıyor.")\n        require(not (folder / "validation/summary.json").is_file(),\n                "Bu çalışmanın doğrulaması bitmiş; sonuçlara bak ya da yeni deney kimliği kullan.")\n        require(zipfile.is_zipfile(checkpoint), "Checkpoint eksik ya da beklenmeyen biçimde.")\n        with zipfile.ZipFile(checkpoint) as archive:\n            require(archive.testzip() is None, "Checkpoint bütünlük kontrolünden geçmedi.")\n    else:\n        require(not folder.exists(), "Çalışma klasörü var. Devam seçeneğini ya da yeni deney kimliğini kullan.")\n    return [str(executable), DATASET_ID, "3d_fullres", "0", "-tr", trainer, "-device", "cuda"] + (["--c"] if resume else [])\n\n\ndef validation_complete(fold_directory, expected_cases):\n    """True only when the final 3d_fullres validation covered every validation case."""\n    summary = Path(fold_directory) / "validation" / "summary.json"\n    if not summary.is_file():\n        return False\n    metrics = json.loads(summary.read_text())\n    return len(metrics.get("metric_per_case", [])) == expected_cases\n\n\ndef experiment_id_ok(value):\n    return re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", value) is not None\n\n\ndef preview(dataset, cases, destination):\n    """Mid-plane overlays drawn in voxel order, so differently stored cases look different here."""\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    from matplotlib.colors import ListedColormap\n    import nibabel as nib\n    import numpy as np\n\n    colors = ListedColormap([(0, 0, 0, 0), (.99, .65, .52, .45), (1, .94, .70, .5), (.4, .64, 1, .5),\n                             (.65, .32, .8, .5), (.9, .9, .9, .5)])\n    figure, axes = plt.subplots(len(cases), 3, figsize=(12, 3.6 * len(cases)), squeeze=False)\n    for row, case in enumerate(cases):\n        image = nib.load(Path(dataset) / "imagesTr" / f"{case}_0000.nii.gz")\n        data = np.asanyarray(image.dataobj).astype(np.float32)\n        label = np.asanyarray(nib.load(Path(dataset) / "labelsTr" / f"{case}.nii.gz").dataobj)\n        low, high = np.percentile(data[data > 0], [1, 99]) if (data > 0).any() else (0, 1)\n        for axis in range(3):\n            index = [slice(None)] * 3\n            index[axis] = data.shape[axis] // 2\n            ax = axes[row, axis]\n            ax.imshow(np.rot90(data[tuple(index)]), cmap="gray", vmin=low, vmax=high)\n            ax.imshow(np.rot90(label[tuple(index)]), cmap=colors, vmin=0, vmax=5, interpolation="nearest")\n            ax.set_title(f"{case} · {\'\'.join(nib.aff2axcodes(image.affine))} · eksen {axis}", fontsize=9)\n            ax.axis("off")\n    figure.tight_layout()\n    figure.savefig(destination, dpi=90)\n    plt.close(figure)\n\n\ndef main():\n    import argparse\n    parser = argparse.ArgumentParser(description="E2 handoff checks for the Colab notebook.")\n    commands = parser.add_subparsers(dest="command", required=True)\n    check = commands.add_parser("audit")\n    check.add_argument("dataset", type=Path)\n    check.add_argument("splits", type=Path)\n    check.add_argument("contract", type=Path)\n    check.add_argument("out", type=Path)\n    ras = commands.add_parser("ras")\n    ras.add_argument("dataset", type=Path)\n    ras.add_argument("raw_out", type=Path)\n    ras.add_argument("out", type=Path)\n    view = commands.add_parser("preview")\n    view.add_argument("dataset", type=Path)\n    view.add_argument("out", type=Path)\n    view.add_argument("cases", nargs="+")\n    args = parser.parse_args()\n    if args.command == "audit":\n        report = audit(args.dataset, args.splits, json.loads(args.contract.read_text()))\n        args.out.write_text(json.dumps(report, indent=2) + "\\n")\n        print(f"{report[\'train_count\']} eğitim / {report[\'validation_count\']} doğrulama; eksen sıraları: {report[\'axis_codes\']}")\n    elif args.command == "ras":\n        changes = write_ras_dataset(args.dataset, args.raw_out)\n        args.out.write_text(json.dumps(changes, indent=2) + "\\n")\n        moved = sorted(case for case, change in changes.items() if change["source_axis_codes"] != "RAS")\n        print(f"RAS\'a çevrilen vaka: {len(moved)} / {len(changes)}")\n    else:\n        preview(args.dataset, args.cases, args.out)\n        print(args.out)\n\n\nif __name__ == "__main__":\n    main()\n', 'trainers_e2.py': '"""nnU-Net 2.8.1 E2 variants: the E1 settings (seed 42, a checkpoint every epoch) on the E2 dataset."""\n\nimport random\nimport numpy as np\nimport torch\n\nfrom nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import (\n    nnUNetTrainer_5epochs, nnUNetTrainer_100epochs,\n)\n\n\nclass nnUNetTrainer_E2Smoke(nnUNetTrainer_5epochs):\n    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):\n        random.seed(42)\n        np.random.seed(42)\n        torch.manual_seed(42)\n        super().__init__(plans, configuration, fold, dataset_json, device)\n        self.save_every = 1\n\n\nclass nnUNetTrainer_E2Preliminary(nnUNetTrainer_100epochs):\n    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):\n        random.seed(42)\n        np.random.seed(42)\n        torch.manual_seed(42)\n        super().__init__(plans, configuration, fold, dataset_json, device)\n        self.save_every = 1\n', 'e2_contract.json': '{\n  "dataset": "Dataset002_E2Charm",\n  "archive": "handoff_k2_e2.zip",\n  "archive_sha256": "aa5489712b83111bcd7cd163bb6d43537a0f49590e1971d8f48e9ba55c8c6fe7",\n  "labels": {\n    "background": 0,\n    "skin": 1,\n    "skull": 2,\n    "csf": 3,\n    "gray_matter": 4,\n    "white_matter": 5\n  },\n  "sources": {\n    "IXI": {\n      "train": 23,\n      "val": 6\n    },\n    "MultiAxial": {\n      "train": 44,\n      "val": 10\n    }\n  },\n  "train": [\n    "GU002",\n    "GU006",\n    "GU008",\n    "GU009",\n    "GU010",\n    "GU015",\n    "GU016",\n    "GU017",\n    "GU019",\n    "GU020",\n    "GU021",\n    "GU024",\n    "GU025",\n    "GU027",\n    "GU030",\n    "GU032",\n    "GU035",\n    "GU036",\n    "GU038",\n    "GU039",\n    "GU040",\n    "IXI028",\n    "IXI037",\n    "IXI045",\n    "IXI054",\n    "IXI062",\n    "IXI070",\n    "IXI078",\n    "IXI087",\n    "IXI095",\n    "IXI103",\n    "IXI185",\n    "IXI195",\n    "IXI212",\n    "IXI222",\n    "IXI239",\n    "IXI249",\n    "IXI329",\n    "IXI377",\n    "IXI385",\n    "IXI393",\n    "IXI409",\n    "IXI417",\n    "IXI442",\n    "NC004",\n    "NC008",\n    "NC010",\n    "NC011",\n    "NC013",\n    "NC014",\n    "NC016",\n    "NC017",\n    "NC018",\n    "NC020",\n    "NC021",\n    "NC022",\n    "NC024",\n    "NC027",\n    "NC031",\n    "NC033",\n    "NYU002",\n    "NYU003",\n    "NYU004",\n    "NYU006",\n    "NYU007",\n    "NYU008",\n    "NYU010"\n  ],\n  "val": [\n    "GU011",\n    "GU012",\n    "GU018",\n    "GU033",\n    "GU041",\n    "IXI148",\n    "IXI167",\n    "IXI294",\n    "IXI320",\n    "IXI368",\n    "IXI426",\n    "NC012",\n    "NC019",\n    "NC023",\n    "NC029",\n    "NYU005"\n  ],\n  "source": "K1 handoff of 2026-11-05 (EXP-002 note); archive SHA-256 from handoff_k2_e2.zip.sha256; split recomputed with tools/nnunet_prep/prepare_e2.plan_cases, seed 42"\n}\n', 'colab_helpers.py': '"""Small reusable Colab helpers; importing this module never starts training."""\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport signal\nimport stat\nimport subprocess\nimport tempfile\nimport time\nimport zipfile\n\nDATASET = "Dataset001_MultiAxial"\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef gpu_probe():\n    import importlib.metadata\n    import torch\n    require(importlib.metadata.version("nnunetv2") == "2.8.1", "Expected nnU-Net 2.8.1.")\n    require(torch.__version__.split("+")[0] == "2.6.0" and torch.version.cuda == "12.4", "Expected torch 2.6.0 / CUDA 12.4.")\n    require(torch.cuda.is_available(), "Choose a GPU runtime in Colab; CUDA is unavailable.")\n    props = torch.cuda.get_device_properties(0)\n    require(props.total_memory >= 12 * 1024**3, "This handoff requests a GPU with at least 12 GB VRAM.")\n    layer = torch.nn.Conv3d(1, 2, 3).cuda()\n    value = torch.randn(1, 1, 12, 12, 12, device="cuda")\n    with torch.autocast("cuda"):\n        result = layer(value).square().mean()\n    result.backward()\n    torch.cuda.synchronize()\n    del layer, value, result\n    torch.cuda.empty_cache()\n    return {"gpu": props.name, "vram_gib": props.total_memory / 1024**3,\n            "torch": torch.__version__, "cuda": torch.version.cuda,\n            "nnunetv2": importlib.metadata.version("nnunetv2"), "conv3d_backward_passed": True}\n\n\ndef pack_handoff(root, report, destination):\n    root, destination = Path(root).resolve(), Path(destination).resolve()\n    require(not destination.exists(), "Archive already exists; choose a new output filename.")\n    require(not any((parent / ".git").exists() for parent in destination.parents), "Keep MRI archives outside Git.")\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    manifest = {"files": report["files"], "dataset_sha256": report["dataset_sha256"]}\n    partial = destination.with_suffix(destination.suffix + ".partial")\n    try:\n        with zipfile.ZipFile(partial, "x", compression=zipfile.ZIP_STORED) as archive:\n            for relative, digest in sorted(manifest["files"].items()):\n                source = root / relative\n                require(sha256(source) == digest, f"Input changed after audit: {relative}")\n                archive.write(source, relative)\n            archive.writestr("expected_files.json", json.dumps(manifest, indent=2) + "\\n")\n        partial.rename(destination)\n    except BaseException:\n        partial.unlink(missing_ok=True)\n        raise\n    return sha256(destination)\n\n\ndef extract_handoff(archive_path, destination):\n    archive_path, destination = Path(archive_path), Path(destination).resolve()\n    archive_digest = sha256(archive_path)\n    marker = destination / ".archive_sha256"\n    if destination.exists():\n        require(marker.is_file() and marker.read_text().strip() == archive_digest,\n                "Existing extraction belongs to another archive. Choose a fresh destination.")\n        return destination\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive_path) as archive:\n        names = archive.namelist()\n        require(len(names) == len(set(names)), "Archive has duplicate paths.")\n        require("expected_files.json" in names and archive.getinfo("expected_files.json").file_size < 2_000_000,\n                "Use the prepared handoff_k2_e1.zip with its checksum manifest.")\n        manifest = json.loads(archive.read("expected_files.json"))\n        expected = manifest["files"]\n        require(set(names) == set(expected) | {"expected_files.json"}, "Archive contains unexpected or missing entries.")\n        require(sum(info.file_size for info in archive.infolist()) <= 20_000_000_000, "Archive is too large.")\n        for info in archive.infolist():\n            path = PurePosixPath(info.filename)\n            require(not path.is_absolute() and ".." not in path.parts and "\\\\" not in info.filename,\n                    "Unsafe archive path.")\n            require(not stat.S_ISLNK(info.external_attr >> 16) and not info.is_dir(), "Archive links/directories are not supported.")\n            allowed = (info.filename in ("splits_final.json", "expected_files.json", f"nnUNet_raw/{DATASET}/dataset.json")\n                       or (path.parent.as_posix() in (f"nnUNet_raw/{DATASET}/imagesTr", f"nnUNet_raw/{DATASET}/labelsTr")\n                           and path.name.endswith(".nii.gz")))\n            require(allowed, "Archive contains files outside the E1 dataset.")\n        with tempfile.TemporaryDirectory(prefix="e1-extract-", dir=destination.parent) as folder:\n            temporary = Path(folder)\n            for info in archive.infolist():\n                output = temporary / info.filename\n                output.parent.mkdir(parents=True, exist_ok=True)\n                with archive.open(info) as source, output.open("xb") as target:\n                    shutil.copyfileobj(source, target, length=1024 * 1024)\n                if info.filename in expected:\n                    require(sha256(output) == expected[info.filename], f"Archive checksum mismatch: {info.filename}")\n            (temporary / ".archive_sha256").write_text(archive_digest + "\\n")\n            temporary.rename(destination)\n    return destination\n\n\ndef verify_manifest(root, report):\n    manifest = json.loads((Path(root) / "expected_files.json").read_text())\n    require(report["files"] == manifest["files"] and report["dataset_sha256"] == manifest["dataset_sha256"],\n            "Extracted data differs from the audited handoff manifest.")\n\n\ndef install_split(root, preprocessed):\n    source = Path(root) / "splits_final.json"\n    target = Path(preprocessed) / DATASET / "splits_final.json"\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if target.exists():\n        require(json.loads(target.read_text()) == json.loads(source.read_text()), "Existing split differs; refusing to overwrite it.")\n    else:\n        shutil.copyfile(source, target)\n    return sha256(target)\n\n\ndef training_command(executable, trainer, fold_directory, resume=False):\n    require(trainer in ("nnUNetTrainer_E1Smoke", "nnUNetTrainer_E1Preliminary"), "Unexpected E1 trainer.")\n    folder = Path(fold_directory)\n    if resume:\n        checkpoints = [folder / f"checkpoint_{name}.pth" for name in ("final", "latest", "best")]\n        checkpoint = next((path for path in checkpoints if path.is_file()), None)\n        require(checkpoint is not None, "Resume requested but no checkpoint exists; refusing to silently restart.")\n        require(not (folder / "validation/summary.json").is_file(), "Run already has completed validation; inspect results or use a new experiment ID.")\n        require(zipfile.is_zipfile(checkpoint), "Checkpoint is incomplete or uses an unexpected format.")\n        with zipfile.ZipFile(checkpoint) as archive:\n            require(archive.testzip() is None, "Checkpoint archive failed its integrity check.")\n    else:\n        require(not folder.exists(), "Run directory exists. Use resume or a new experiment ID; no overwrite allowed.")\n    return [str(executable), "1", "3d_fullres", "0", "-tr", trainer, "-device", "cuda"] + (["--c"] if resume else [])\n\n\ndef record_run(path, contract):\n    path = Path(path)\n    if path.exists():\n        require(json.loads(path.read_text()) == contract, "Experiment contract changed; use a new experiment ID.")\n    else:\n        path.parent.mkdir(parents=True, exist_ok=True)\n        path.write_text(json.dumps(contract, indent=2) + "\\n")\n\n\ndef run_logged(command, log_path, env=None):\n    """Tail a durable log; stop owned process group when the notebook is interrupted."""\n    log_path = Path(log_path)\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    started = time.monotonic()\n    child_env = dict(os.environ if env is None else env)\n    child_env["PYTHONUNBUFFERED"] = "1"\n    with log_path.open("x", buffering=1) as output:\n        process = subprocess.Popen(command, stdout=output, stderr=subprocess.STDOUT,\n                                   stdin=subprocess.DEVNULL, env=child_env, start_new_session=True)\n        try:\n            with log_path.open(errors="replace") as reader:\n                while True:\n                    text = reader.read()\n                    if text:\n                        print(text, end="", flush=True)\n                    code = process.poll()\n                    if code is not None:\n                        print(reader.read(), end="", flush=True)\n                        if code != 0:\n                            raise subprocess.CalledProcessError(code, command)\n                        return {"elapsed_seconds": time.monotonic() - started, "exit_code": code}\n                    time.sleep(.3)\n        finally:\n            # Colab is Linux; group ownership prevents orphaned augmentation workers.\n            try:\n                os.killpg(process.pid, signal.SIGTERM)\n            except ProcessLookupError:\n                pass\n            if process.poll() is None:\n                try:\n                    process.wait(timeout=10)\n                except subprocess.TimeoutExpired:\n                    os.killpg(process.pid, signal.SIGKILL)\n                    process.wait()\n            try:\n                os.killpg(process.pid, signal.SIGKILL)\n            except ProcessLookupError:\n                pass\n', 'requirements-colab.lock.txt': '# This file was autogenerated by uv via the following command:\n#    uv pip compile tools/e1_training/requirements-colab.txt --python-version 3.11 --python-platform x86_64-unknown-linux-gnu --torch-backend cu124 --output-file /private/tmp/e1-linux-requirements.txt\nacvl-utils==0.2.7\n    # via nnunetv2\nannotated-types==0.8.0\n    # via pydantic\nanyio==4.16.0\n    # via\n    #   httpx\n    #   httpx2\nbatchgenerators==0.25.6\n    # via\n    #   acvl-utils\n    #   batchgeneratorsv2\n    #   nnunetv2\nbatchgeneratorsv2==0.3.7\n    # via nnunetv2\nblosc2==4.14.1\n    # via\n    #   acvl-utils\n    #   nnunetv2\ncertifi==2026.10.23\n    # via\n    #   httpcore\n    #   httpx\n    #   requests\ncharset-normalizer==3.5.2\n    # via requests\nclick==8.5.0\n    # via huggingface-hub\ncloudpickle==3.1.2\n    # via joblib\nconnected-components-3d==4.1.1\n    # via acvl-utils\ncontourpy==1.3.3\n    # via matplotlib\ncycler==0.12.1\n    # via matplotlib\ndynamic-network-architectures==0.4.4\n    # via nnunetv2\neinops==0.8.2\n    # via\n    #   dynamic-network-architectures\n    #   nnunetv2\nfilelock==4.2.7\n    # via\n    #   huggingface-hub\n    #   torch\nfonttools==4.68.0\n    # via matplotlib\nfsspec==2026.9.0\n    # via\n    #   huggingface-hub\n    #   torch\ngraphviz==0.21\n    # via nnunetv2\nh11==0.16.0\n    # via\n    #   httpcore\n    #   httpcore2\nh2==4.4.1\n    # via httpx\nhf-xet==1.8.0\n    # via huggingface-hub\nhpack==4.2.0\n    # via h2\nhttpcore==1.0.9\n    # via httpx\nhttpcore2==2.16.0\n    # via httpx2\nhttpx==0.28.1\n    # via blosc2\nhttpx2==2.16.0\n    # via huggingface-hub\nhuggingface-hub==2.4.0\n    # via timm\nhyperframe==6.1.0\n    # via h2\nidna==3.20\n    # via\n    #   anyio\n    #   httpx\n    #   httpx2\n    #   requests\nimagecodecs==2026.1.14\n    # via nnunetv2\nimageio==2.38.1\n    # via scikit-image\nimportlib-resources==7.1.0\n    # via nibabel\njinja2==3.1.6\n    # via torch\njoblib==1.6.0\n    # via scikit-learn\nkiwisolver==1.5.1\n    # via matplotlib\nlazy-loader==0.6\n    # via scikit-image\nmarkdown-it-py==4.2.0\n    # via rich\nmarkupsafe==3.0.4\n    # via jinja2\nmatplotlib==3.11.2\n    # via\n    #   nnunetv2\n    #   seaborn\nmdurl==0.1.2\n    # via markdown-it-py\nmpmath==1.3.0\n    # via sympy\nmsgpack==1.2.4\n    # via blosc2\nnarwhals==2.28.0\n    # via scikit-learn\nndindex==1.10.1\n    # via blosc2\nnetworkx==3.6.1\n    # via\n    #   scikit-image\n    #   torch\nnibabel==5.4.2\n    # via\n    #   -r tools/e1_training/requirements-colab.txt\n    #   nnunetv2\nnnunetv2==2.8.1\n    # via -r tools/e1_training/requirements-colab.txt\nnumexpr==2.14.2\n    # via blosc2\nnumpy==1.26.4\n    # via\n    #   -r tools/e1_training/requirements-colab.txt\n    #   acvl-utils\n    #   batchgenerators\n    #   batchgeneratorsv2\n    #   blosc2\n    #   connected-components-3d\n    #   contourpy\n    #   dynamic-network-architectures\n    #   imagecodecs\n    #   imageio\n    #   matplotlib\n    #   nibabel\n    #   nnunetv2\n    #   numexpr\n    #   pandas\n    #   scikit-image\n    #   scikit-learn\n    #   scipy\n    #   seaborn\n    #   tifffile\n    #   torchvision\nnvidia-cublas-cu12==12.4.5.8\n    # via\n    #   nvidia-cudnn-cu12\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cuda-cupti-cu12==12.4.127\n    # via torch\nnvidia-cuda-nvrtc-cu12==12.4.127\n    # via torch\nnvidia-cuda-runtime-cu12==12.4.127\n    # via torch\nnvidia-cudnn-cu12==9.1.0.70\n    # via torch\nnvidia-cufft-cu12==11.2.1.3\n    # via torch\nnvidia-curand-cu12==10.3.5.147\n    # via torch\nnvidia-cusolver-cu12==11.6.1.9\n    # via torch\nnvidia-cusparse-cu12==12.3.1.170\n    # via\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cusparselt-cu12==0.6.2\n    # via torch\nnvidia-nccl-cu12==2.21.5\n    # via torch\nnvidia-nvjitlink-cu12==12.4.127\n    # via\n    #   nvidia-cusolver-cu12\n    #   nvidia-cusparse-cu12\n    #   torch\nnvidia-nvtx-cu12==12.4.127\n    # via torch\npackaging==26.3\n    # via\n    #   huggingface-hub\n    #   lazy-loader\n    #   matplotlib\n    #   nibabel\n    #   scikit-image\npandas==3.0.6\n    # via\n    #   batchgenerators\n    #   nnunetv2\n    #   seaborn\npillow==13.0.0\n    # via\n    #   batchgenerators\n    #   imageio\n    #   matplotlib\n    #   scikit-image\n    #   torchvision\npydantic==2.14.2\n    # via blosc2\npydantic-core==2.50.2\n    # via pydantic\npygments==2.21.0\n    # via rich\npyparsing==3.3.3\n    # via matplotlib\npython-dateutil==4.0.0\n    # via\n    #   matplotlib\n    #   pandas\npyyaml==6.0.3\n    # via\n    #   huggingface-hub\n    #   timm\n    #   yacs\nrequests==2.34.2\n    # via nnunetv2\nrich==15.0.0\n    # via blosc2\nsafetensors==0.8.0\n    # via timm\nscikit-image==0.26.0\n    # via\n    #   acvl-utils\n    #   batchgenerators\n    #   nnunetv2\nscikit-learn==1.9.1\n    # via\n    #   batchgenerators\n    #   nnunetv2\nscipy==1.17.1\n    # via\n    #   batchgenerators\n    #   nnunetv2\n    #   scikit-image\n    #   scikit-learn\nseaborn==0.13.2\n    # via nnunetv2\nsimpleitk==2.5.7\n    # via\n    #   acvl-utils\n    #   nnunetv2\nsympy==1.13.1\n    # via torch\nthreadpoolctl==3.7.0\n    # via\n    #   batchgenerators\n    #   blosc2\n    #   scikit-learn\ntifffile==2026.3.3\n    # via\n    #   nnunetv2\n    #   scikit-image\ntimm==1.0.22\n    # via dynamic-network-architectures\ntorch==2.6.0+cu124\n    # via\n    #   -r tools/e1_training/requirements-colab.txt\n    #   acvl-utils\n    #   batchgeneratorsv2\n    #   dynamic-network-architectures\n    #   nnunetv2\n    #   timm\n    #   torchvision\ntorchvision==0.21.0+cu124\n    # via timm\ntqdm==4.70.1\n    # via\n    #   acvl-utils\n    #   huggingface-hub\n    #   nnunetv2\ntriton==3.2.0\n    # via torch\ntruststore==0.10.5\n    # via\n    #   httpcore2\n    #   httpx2\ntyping-extensions==4.16.0\n    # via\n    #   anyio\n    #   httpx2\n    #   huggingface-hub\n    #   nibabel\n    #   pydantic\n    #   pydantic-core\n    #   torch\n    #   typing-inspection\ntyping-inspection==0.4.4\n    # via pydantic\ntzdata==2026.6\n    # via python-dateutil\nurllib3==2.8.0\n    # via requests\nyacs==0.1.8\n    # via nnunetv2\n'}
import hashlib
for name, source in EMBEDDED.items():
    (TOOLS / name).write_text(source)
sys.path.insert(0, str(TOOLS))
from e2_helpers import (extract_handoff, install_split, training_command, validation_complete, require, sha256, DATASET, DATASET_ID)
from colab_helpers import record_run, run_logged
CONTRACT = json.loads(EMBEDDED['e2_contract.json'])
LOCK_SHA256 = sha256(TOOLS / 'requirements-colab.lock.txt')
TRAINER_SHA256 = sha256(TOOLS / 'trainers_e2.py')
print(f"Sözleşme: {len(CONTRACT['train'])} eğitim / {len(CONTRACT['val'])} doğrulama; arşiv {CONTRACT['archive_sha256'][:12]}…")


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.11.8"], check=True)
uv = [sys.executable, "-m", "uv"]
if not PYTHON.is_file():
    subprocess.run(uv + ["venv", str(VENV), "--python", "3.11"], check=True)
subprocess.run(uv + ["pip", "install", "--python", str(PYTHON), "--torch-backend", "cu124",
                     "-r", str(TOOLS / "requirements-colab.lock.txt")], check=True)
subprocess.run(uv + ["pip", "check", "--python", str(PYTHON)], check=True)
trainer_path = Path(subprocess.check_output([str(PYTHON), "-c",
    "from pathlib import Path; import nnunetv2; print(Path(nnunetv2.__file__).parent / 'training/nnUNetTrainer/variants/training_length/nnUNetTrainer_E2.py')"], text=True).strip().splitlines()[-1])
if trainer_path.exists():
    require(trainer_path.read_text() == EMBEDDED["trainers_e2.py"], "Eğitici kodu değişmiş; ortamı ve deney kimliğini kontrol et.")
else:
    trainer_path.write_text(EMBEDDED["trainers_e2.py"])
ENV = os.environ.copy()
ENV.update({"nnUNet_raw": str(RAS_RAW if CANONICALIZE_TO_RAS else DATA), "nnUNet_preprocessed": str(PREPROCESSED),
            "nnUNet_results": str(RESULTS), "nnUNet_n_proc_DA": str(AUGMENTATION_WORKERS),
            "nnUNet_compile": "false", "PYTHONHASHSEED": "42", "MPLBACKEND": "Agg",
            "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"})
require(subprocess.check_output([str(PYTHON), "-c", "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"], text=True).strip() == "3.11", "Python sürümü 3.11 olmalı.")
gpu = json.loads(subprocess.check_output([str(PYTHON), "-c",
    "import sys,json; sys.path.insert(0, '/content/e2_tools'); from colab_helpers import gpu_probe; print(json.dumps(gpu_probe()))"], env=ENV, text=True))
print(json.dumps(gpu, indent=2))
session_id = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
(RESULTS / f"session_{session_id}.json").write_text(json.dumps({**gpu, "cu_per_hour": CU_PER_HOUR, "available_cu": AVAILABLE_CU}, indent=2))
(RESULTS / f"environment_{session_id}.txt").write_text(subprocess.check_output(uv + ["pip", "freeze", "--python", str(PYTHON)], text=True))


## 3 · Paketi doğrula ve tek eksen sırasına getir
Arşivin SHA-256 değeri K1'in yayımladığı değerle aynı olmalı. Sonra bölme K1'in atamasından hesaplanan
67/16 listesiyle, dosya adları, ızgaralar ve 0–5 etiketleri tek tek denetlenir.

**Eksen sırası:** IXI görüntüleri PSR (sagital), MultiAxial görüntüleri RAS saklanıyor. nnU-Net görüntüyü
döndürmez, yani ağ iki farklı dizi düzeni görür. `CANONICALIZE_TO_RAS = True` her vakayı RAS sırasına
getirir: yalnız eksen değiştirme/çevirme, interpolasyon yok, her vokselin dünya konumu aynı kalır
(K3 koordinat sözleşmesi, D-004). Bu modelle çıkarım yapılırken PSR girdiler de aynı şekilde RAS'a çevrilmelidir
(`tools/headmodel/model_io.py prepare/restore`).


In [ ]:
layout = extract_handoff(ARCHIVE, DATA, CONTRACT["archive_sha256"])
contract_path = TOOLS / "e2_contract.json"
stamp = str(time.time_ns())
run_logged([str(PYTHON), str(TOOLS / "e2_helpers.py"), "audit", str(layout["dataset"]), str(layout["splits"]),
            str(contract_path), str(WORK / "audit_source.json")], RESULTS / "logs" / f"audit_source_{stamp}.log", ENV)
source_report = json.loads((WORK / "audit_source.json").read_text())
shutil.copyfile(WORK / "audit_source.json", RESULTS / f"audit_source_{stamp}.json")
for warning in source_report["warnings"]:
    print("UYARI:", warning)
if CANONICALIZE_TO_RAS:
    if not (RAS_RAW / DATASET).exists():
        run_logged([str(PYTHON), str(TOOLS / "e2_helpers.py"), "ras", str(layout["dataset"]), str(RAS_RAW),
                    str(WORK / "orientation_changes.json")], RESULTS / "logs" / f"ras_{stamp}.log", ENV)
    shutil.copyfile(WORK / "orientation_changes.json", RESULTS / f"orientation_changes_{stamp}.json")
    TRAIN_DATASET = RAS_RAW / DATASET
else:
    TRAIN_DATASET = layout["dataset"]
run_logged([str(PYTHON), str(TOOLS / "e2_helpers.py"), "audit", str(TRAIN_DATASET), str(layout["splits"]),
            str(contract_path), str(WORK / "audit_train.json")], RESULTS / "logs" / f"audit_train_{stamp}.log", ENV)
report = json.loads((WORK / "audit_train.json").read_text())
shutil.copyfile(WORK / "audit_train.json", RESULTS / f"audit_train_{stamp}.json")
require(report["train_count"] == len(CONTRACT["train"]) and report["validation_count"] == len(CONTRACT["val"]), "Bölme değişmiş.")
require(not CANONICALIZE_TO_RAS or report["axis_codes"] == {"RAS": len(CONTRACT["train"]) + len(CONTRACT["val"])},
        "RAS kopyasında RAS olmayan vaka kaldı.")
print("Eğitimde kullanılacak eksen sıraları:", report["axis_codes"])


In [ ]:
# Yalnız eğitim vakaları; voksel düzleminde önizleme. Hepsi aynı yönde görünmeli.
train_ids = CONTRACT["train"]
shown = [c for c in train_ids if c.startswith("IXI")][:2] + [c for c in train_ids if not c.startswith("IXI")][:2]
preview = RESULTS / "e2_label_review.png"
subprocess.run([str(PYTHON), str(TOOLS / "e2_helpers.py"), "preview", str(TRAIN_DATASET), str(preview), *shown], check=True, env=ENV)
from IPython.display import display, Image
display(Image(filename=str(preview)))


**Veri inceleme noktası.** Etiketler charm 4.6.0 çıktısıdır (elle çizilmiş referans değil); kan BOS'a,
sıfır dolgu üzerindeki etiketler arka plana çevrildi; MultiAxial charm koşuları yalnız T1 ile yapıldı
(kafatası T1+T2'ye göre zayıf olabilir); 19 MultiAxial vakada lezyon kuralı uygulandı (EXP-002 notu).
Doğrulama Dice'ı charm etiketlerine göre hesaplanır; E1'in MultiAxial referansına göre Dice'ı ile doğrudan
karşılaştırılamaz.

Önizlemeyi ve bu notları inceleyip devam etmeyi seçersen ilk hücrede `ACKNOWLEDGE_E2_DATA = True` yap
(ya da aşağıya `ACKNOWLEDGE_E2_DATA = True` yazıp çalıştır). Bu, anatomik onay anlamına gelmez.


## 4 · Ön işleme ve sabit fold 0
K1 masaüstündeki plan: 1 mm, patch 160×128×112, batch 2 (RAS'a çevirmeden önceki eksen sırasıyla).
Burada bu ortamın ürettiği plan gösterilir. Colab yeniden başlarsa yerel ön işleme yeniden yapılır;
Drive'daki ağırlıklar silinmez.


In [ ]:
require(ACKNOWLEDGE_E2_DATA, "Önizlemeyi ve veri notlarını incele; sonra ACKNOWLEDGE_E2_DATA = True yap.")
marker = PREPROCESSED / ".e2_input_sha256"
preprocessing_key = hashlib.sha256((report["dataset_sha256"] + LOCK_SHA256 + str(CANONICALIZE_TO_RAS)).encode()).hexdigest()
if marker.is_file():
    require(marker.read_text().strip() == preprocessing_key, "Ön işleme başka veri/sürüme ait; yeni yerel klasör kullan.")
else:
    run_logged([str(VENV / "bin/nnUNetv2_plan_and_preprocess"), "-d", DATASET_ID, "-c", "3d_fullres",
                "--verify_dataset_integrity", "-np", "1"], RESULTS / "logs" / f"preprocess_{time.time_ns()}.log", ENV)
    marker.parent.mkdir(parents=True, exist_ok=True)
    marker.write_text(preprocessing_key + "\n")
split_hash = install_split(layout["splits"], PREPROCESSED)
require(split_hash == report["split_sha256"], "Sabit bölme kopyası değişmiş.")
plan_path = PREPROCESSED / DATASET / "nnUNetPlans.json"
plans = json.loads(plan_path.read_text())
require("3d_fullres" in plans["configurations"], "3d_fullres planı oluşmadı.")
print(json.dumps(plans["configurations"]["3d_fullres"], indent=2))
contract = {"experiment": "E2", "archive_sha256": CONTRACT["archive_sha256"],
            "source_dataset_sha256": source_report["dataset_sha256"], "dataset_sha256": report["dataset_sha256"],
            "canonicalized_to_ras": CANONICALIZE_TO_RAS, "split_sha256": split_hash, "plans_sha256": sha256(plan_path),
            "lock_sha256": LOCK_SHA256, "trainer_sha256": TRAINER_SHA256, "configuration": "3d_fullres", "fold": 0,
            "nnunetv2": "2.8.1", "torch": "2.6.0+cu124", "seed": 42, "augmentation_workers": AUGMENTATION_WORKERS,
            "torch_compile": False, "preliminary_only": True, "test_accessed": False}
record_run(RESULTS / "experiment_contract.json", contract)
for path in (plan_path, PREPROCESSED / DATASET / "dataset_fingerprint.json", layout["splits"]):
    target = RESULTS / path.name
    if target.exists():
        require(sha256(target) == sha256(path), f"Kayıtlı {path.name} değişmiş; yeni deney kimliği kullan.")
    else:
        shutil.copyfile(path, target)


## 5 · İsteğe bağlı 5 epoch kontrol
Mert'in notu: bu adım atlanabilir. Açarsan (`RUN_5_EPOCH_PILOT = True`) süreyi ölçer ve 6. bölümde 100 epoch tahmini verir.
Her epoch'ta checkpoint alınır. Devam için aynı deney kimliği ve `RESUME_PILOT = True`.


In [ ]:
def train_stage(trainer, enabled, resume):
    if not enabled:
        print(f"{trainer}: kapalı; eğitim başlatılmadı.")
        return None
    require(ACKNOWLEDGE_E2_DATA and (RESULTS / "experiment_contract.json").is_file(), "Veri/ön işleme adımları tamamlanmalı.")
    require(os.path.ismount("/content/drive"), "Checkpoint hedefi kalıcı Drive bağlantısında olmalı.")
    require(ENV["nnUNet_results"] == str(RESULTS) and ENV["nnUNet_n_proc_DA"] == str(AUGMENTATION_WORKERS),
            "Ayarlar değişti; ortam/veri/ön işleme hücrelerini yeniden çalıştır.")
    record_run(RESULTS / "experiment_contract.json", contract)
    folder = RESULTS / DATASET / f"{trainer}__nnUNetPlans__3d_fullres" / "fold_0"
    command = training_command(VENV / "bin/nnUNetv2_train", trainer, folder, resume)
    log = RESULTS / "logs" / f"{trainer}_{time.time_ns()}.log"
    result = run_logged(command, log, ENV)
    (log.with_suffix(".json")).write_text(json.dumps(result, indent=2))
    print(f"Bitti: {result['elapsed_seconds']/60:.1f} dakika. Log: {log}")
    return folder
train_stage("nnUNetTrainer_E2Smoke", RUN_5_EPOCH_PILOT, RESUME_PILOT)


## 6 · Süre/birim tahmini (5 epoch çalıştıysa)
İlk epoch dışarıda bırakılıp sonraki epoch sürelerinin medyanı kullanılır. Kurulum, ön işleme ve son doğrulama dahil değildir.


In [ ]:
logs = sorted((RESULTS / "logs").glob("nnUNetTrainer_E2Smoke_*.log"))
epoch_seconds = []
for log in logs:
    epoch_seconds.extend(float(value) for value in re.findall(r"Epoch time:\s*([0-9.]+)", log.read_text()))
if len(epoch_seconds) >= 2:
    import statistics
    hours = statistics.median(epoch_seconds[1:]) * 100 / 3600
    print(f"100 epoch eğitim döngüsü yaklaşık {hours:.2f} saat; ek aşamalar dahil değil.")
    if CU_PER_HOUR > 0:
        print(f"Girilen oranla yaklaşık {hours * CU_PER_HOUR:.1f} işlem birimi + ek aşamalar.")
else:
    print("Tahmin için 5 epoch kontrolünden en az iki epoch süresi gerekli (adım atlandıysa normal).")


## 7 · 100 epoch E2 ön modeli — ana eğitim
`RUN_100_EPOCH_PRELIMINARY = True` yap (ya da aşağıdaki hücrenin başına yaz) ve çalıştır. Sıfırdan başlar;
5 epoch ağırlıklarını kullanmaz. Oturum koparsa aynı deney kimliğiyle `RESUME_PRELIMINARY = True` ver;
son checkpoint'ten devam eder. `RELEASE_RUNTIME_WHEN_DONE = True` ise eğitim ve tam doğrulama bitince
sonuç Drive'a yazılır ve GPU oturumu bırakılır.


In [ ]:
final_folder = train_stage("nnUNetTrainer_E2Preliminary", RUN_100_EPOCH_PRELIMINARY, RESUME_PRELIMINARY)
if final_folder is not None and RELEASE_RUNTIME_WHEN_DONE:
    checkpoint = final_folder / "checkpoint_final.pth"
    require(checkpoint.is_file() and checkpoint.stat().st_size > 0, "Son checkpoint yok; oturum açık bırakıldı.")
    require(validation_complete(final_folder, 16), "Doğrulama 16 vakayı kapsamıyor; oturum açık bırakıldı.")
    metrics = json.loads((final_folder / "validation/summary.json").read_text())
    (RESULTS / "run_success.json").write_text(json.dumps({"checkpoint": str(checkpoint), "validation_cases": 16,
        "foreground_mean": metrics["foreground_mean"], "runtime_release_requested": True}, indent=2) + "\n")
    print("Sonuçlar Drive'a kaydedildi:", metrics["foreground_mean"], flush=True)
    from google.colab import runtime
    runtime.unassign()


## 8 · Doğrulama çıktıları
Epoch logundaki pseudo-Dice ile 16 doğrulama vakasının son segmentasyon Dice'ı farklıdır.
Bu Dice charm etiketlerine göredir. Test kümesi bu defterde açılmaz.


In [ ]:
folder = RESULTS / DATASET / "nnUNetTrainer_E2Preliminary__nnUNetPlans__3d_fullres/fold_0"
if validation_complete(folder, 16):
    metrics = json.loads((folder / "validation/summary.json").read_text())
    print("Yalnız 16 doğrulama vakası — ön sonuç (charm etiketlerine göre):")
    print(json.dumps({"foreground_mean": metrics.get("foreground_mean"), "per_label_mean": metrics.get("mean")}, indent=2))
else:
    print("Tamamlanmış 100 epoch doğrulama özeti henüz yok.")
print("Ara verirken checkpoint/logların Drive'da olduğunu kontrol et; kullanılmayan GPU oturumunu kapat.")


Kaynaklar: [nnU-Net v2.8.1](https://github.com/MIC-DKFZ/nnUNet/tree/v2.8.1) ·
E2 veri hazırlığı `tools/nnunet_prep/prepare_e2.py` · E1 akışı `tools/e1_training/build_notebook.py` ·
bu defterin üreticisi `tools/e2_training/build_e2_notebook.py`.
